### Blood Report Extraction - MinIO PDF → PostgreSQL

**Pipeline:**
```
MinIO (PDF)
  → Download temporarily (Both Tabula and Tesseract are local processing libraries,only know how to read files from machine's disk.)
  → Tabula  (digital PDFs — extracts tables)
  → Tesseract (scanned PDFs — OCR fallback)
  → Parse key markers (glucose, hemoglobin, cholesterol …)
  → Validate values
  → Store in PostgreSQL blood_reports table
```

**Duplicate prevention:** `(customer_id, report_date, lab_ref)` unique constraint — re-running is always safe.

#### 1. Install dependencies

In [ ]:
!pip install minio psycopg2-binary python-dotenv tabula-py pytesseract pdf2image pillow pandas --quiet

# Tesseract binary must also be installed on your machine:
# Windows  → https://github.com/UB-Mannheim/tesseract/wiki  (add to PATH)
# Mac      → brew install tesseract
# Linux    → sudo apt install tesseract-ocr

In [ ]:
!pip install pdfplumber --quiet

#### 2. Imports & connections

In [ ]:
import os
import re
import json
import tempfile
import psycopg2
import tabula
import pytesseract
import pandas as pd
from minio import Minio
from pdf2image import convert_from_path
from dotenv import load_dotenv
import pdfplumber
from datetime import datetime, timezone
load_dotenv()



True

In [2]:
# ── clients ──────────────────────────────────────────────
def get_minio():
    return Minio(
        "localhost:9000",
        access_key=os.getenv("MINIO_ROOT_USER"),
        secret_key=os.getenv("MINIO_ROOT_PASSWORD"),
        secure=False
    )

def get_pg():
    return psycopg2.connect(
        host="localhost", port=5432,
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD")
    )

minio = get_minio()
pg    = get_pg()
print("✅ Connected")

✅ Connected


#### 3. Create `blood_reports` table (run once)

In [ ]:
# CREATE_TABLE = """
# CREATE TABLE IF NOT EXISTS blood_reports (
#     id                  SERIAL PRIMARY KEY,

#     -- identity
#     customer_id         TEXT NOT NULL,
#     report_date         DATE,
#     lab_name            TEXT,
#     lab_ref             TEXT,

#     -- key blood markers (NULL = not found in report)
#     glucose             DECIMAL(6,2),   -- mg/dL  normal: 70–100
#     hemoglobin          DECIMAL(5,2),   -- g/dL   normal: 12–17
#     cholesterol_total   DECIMAL(6,2),   -- mg/dL  normal: <200
#     cholesterol_hdl     DECIMAL(6,2),   -- mg/dL  normal: >40
#     cholesterol_ldl     DECIMAL(6,2),   -- mg/dL  normal: <100
#     triglycerides       DECIMAL(6,2),   -- mg/dL  normal: <150
#     wbc                 DECIMAL(6,2),   -- x10³/µL normal: 4–11
#     rbc                 DECIMAL(6,2),   -- x10⁶/µL normal: 4.5–5.5

#     -- flexible storage for all other markers
#     raw_data            JSONB,

#     -- traceability
#     minio_path          TEXT,
#     extraction_method   TEXT,           -- tabula | tesseract
#     is_valid            BOOLEAN DEFAULT TRUE,
#     validation_notes    TEXT,
#     indexed_at          TIMESTAMPTZ DEFAULT NOW(),

#     -- prevent duplicate records
#     UNIQUE (customer_id, report_date, lab_ref)
# );
# """

# with pg.cursor() as cur:
#     cur.execute(CREATE_TABLE)
# pg.commit()
# print("✅ Table ready")

In [29]:
CREATE_TABLE = """
CREATE TABLE IF NOT EXISTS blood_reports (
    id                  SERIAL PRIMARY KEY,

    -- identity
    customer_id         TEXT        NOT NULL,
    report_date         DATE        NOT NULL,
    lab_ref             TEXT,

    -- patient demographics (useful for range validation)
    patient_name        TEXT,
    patient_dob         DATE,
    patient_gender      TEXT        CHECK (patient_gender IN ('male', 'female', 'other')),
    patient_age         INT,
    condition_note      TEXT,                      -- e.g. 'Healthy', 'Diabetic'

    -- CBC markers
    hemoglobin          DECIMAL(5,2),              -- g/dL        normal M: 13.5–17.5, F: 12.0–15.5
    wbc                 DECIMAL(6,2),              -- ×10³/μL     normal: 4.5–11.0
    rbc                 DECIMAL(6,2),              -- ×10⁶/μL     normal M: 4.5–5.9, F: 4.0–5.2
    platelets           DECIMAL(7,2),              -- ×10³/μL     normal: 150–400
    hematocrit          DECIMAL(5,2),              -- %           normal M: 41–53, F: 36–46
    mcv                 DECIMAL(6,2),              -- fL          normal: 80–100
    mch                 DECIMAL(5,2),              -- pg          normal: 27–33
    mchc                DECIMAL(5,2),              -- g/dL        normal: 32–36

    -- metabolic / glucose markers
    glucose             DECIMAL(6,2),              -- mg/dL       normal fasting: 70–100
    hba1c               DECIMAL(5,2),              -- %           normal: 4.0–5.6

    -- kidney markers
    creatinine          DECIMAL(5,3),              -- mg/dL       normal M: 0.6–1.2, F: 0.5–1.1
    bun                 DECIMAL(6,2),              -- mg/dL       normal: 7–20
    egfr                DECIMAL(6,2),              -- mL/min      normal: >60

    -- lipid panel
    cholesterol_total   DECIMAL(6,2),              -- mg/dL       normal: <200
    cholesterol_hdl     DECIMAL(6,2),              -- mg/dL       normal M: >40, F: >50
    cholesterol_ldl     DECIMAL(6,2),              -- mg/dL       normal: <100
    triglycerides       DECIMAL(6,2),              -- mg/dL       normal: <150

    -- liver markers
    alt                 DECIMAL(6,2),              -- U/L         normal: 7–56
    ast                 DECIMAL(6,2),              -- U/L         normal: 10–40
    bilirubin_total     DECIMAL(5,2),              -- mg/dL       normal: 0.2–1.2

    -- thyroid
    tsh                 DECIMAL(6,3),              -- mIU/L       normal: 0.4–4.0

    -- flexible storage for unlisted markers
    raw_data            JSONB,


    -- traceability
    minio_path          TEXT,
    extraction_method   TEXT        CHECK (extraction_method IN ('tabula', 'tesseract', 'manual','pdfplumber','api')),
    is_valid            BOOLEAN     DEFAULT TRUE,
    validation_notes    TEXT,
    updated_at          TIMESTAMPTZ DEFAULT NOW(),

    -- prevent duplicate records
    UNIQUE (customer_id, report_date, lab_ref)
);
"""

with pg.cursor() as cur:
    cur.execute(CREATE_TABLE)
pg.commit()
print("✅ Table ready")

✅ Table ready


In [30]:
# def drop_table(pg):
#     # close any open transaction first
#     pg.rollback()
#     pg.autocommit = True

#     with pg.cursor() as cur:
        

#         cur.execute("DROP TABLE IF EXISTS blood_reports CASCADE;")
#         print("  🗑️  Dropped table → blood_report")

#     pg.autocommit = False
    

# drop_table(pg)

In [ ]:
# CREATE TABLE IF NOT EXISTS blood_reports (
#     id                  SERIAL PRIMARY KEY,

#     -- identity
#     customer_id         TEXT        NOT NULL,
#     report_date         DATE        NOT NULL,
#     lab_ref             TEXT,

#     -- patient demographics (useful for range validation)
#     patient_name        TEXT,
#     patient_dob         DATE,
#     patient_gender      TEXT        CHECK (patient_gender IN ('male', 'female', 'other')),
#     patient_age         INT,
#     condition_note      TEXT,                      -- e.g. 'Healthy', 'Diabetic'

#     -- CBC markers
#     hemoglobin          DECIMAL(5,2),              -- g/dL        normal M: 13.5–17.5, F: 12.0–15.5
#     wbc                 DECIMAL(6,2),              -- ×10³/μL     normal: 4.5–11.0
#     rbc                 DECIMAL(6,2),              -- ×10⁶/μL     normal M: 4.5–5.9, F: 4.0–5.2
#     platelets           DECIMAL(7,2),              -- ×10³/μL     normal: 150–400
#     hematocrit          DECIMAL(5,2),              -- %           normal M: 41–53, F: 36–46
#     mcv                 DECIMAL(6,2),              -- fL          normal: 80–100
#     mch                 DECIMAL(5,2),              -- pg          normal: 27–33
#     mchc                DECIMAL(5,2),              -- g/dL        normal: 32–36

#     -- metabolic / glucose markers
#     glucose             DECIMAL(6,2),              -- mg/dL       normal fasting: 70–100
#     hba1c               DECIMAL(5,2),              -- %           normal: 4.0–5.6

#     -- kidney markers
#     creatinine          DECIMAL(5,3),              -- mg/dL       normal M: 0.6–1.2, F: 0.5–1.1
#     bun                 DECIMAL(6,2),              -- mg/dL       normal: 7–20
#     egfr                DECIMAL(6,2),              -- mL/min      normal: >60

#     -- lipid panel
#     cholesterol_total   DECIMAL(6,2),              -- mg/dL       normal: <200
#     cholesterol_hdl     DECIMAL(6,2),              -- mg/dL       normal M: >40, F: >50
#     cholesterol_ldl     DECIMAL(6,2),              -- mg/dL       normal: <100
#     triglycerides       DECIMAL(6,2),              -- mg/dL       normal: <150

#     -- liver markers
#     alt                 DECIMAL(6,2),              -- U/L         normal: 7–56
#     ast                 DECIMAL(6,2),              -- U/L         normal: 10–40
#     bilirubin_total     DECIMAL(5,2),              -- mg/dL       normal: 0.2–1.2

#     -- thyroid
#     tsh                 DECIMAL(6,3),              -- mIU/L       normal: 0.4–4.0

#     -- flexible storage for unlisted markers
#     raw_data            JSONB,

#     -- traceability
#     minio_path          TEXT,
#     extraction_method   TEXT        CHECK (extraction_method IN ('tabula', 'tesseract', 'manual', 'api')),
#     is_valid            BOOLEAN     DEFAULT TRUE,
#     validation_notes    TEXT,
#     indexed_at          TIMESTAMPTZ DEFAULT NOW(),
#     updated_at          TIMESTAMPTZ DEFAULT NOW(),

#     -- deduplication
#     UNIQUE (customer_id, report_date, lab_ref)
# );

# -- indexes for common query patterns
# CREATE INDEX IF NOT EXISTS idx_blood_reports_customer
#     ON blood_reports (customer_id);

# CREATE INDEX IF NOT EXISTS idx_blood_reports_date
#     ON blood_reports (report_date DESC);

# CREATE INDEX IF NOT EXISTS idx_blood_reports_customer_date
#     ON blood_reports (customer_id, report_date DESC);

# CREATE INDEX IF NOT EXISTS idx_blood_reports_raw_data
#     ON blood_reports USING GIN (raw_data);

# -- auto-update updated_at on row changes
# CREATE OR REPLACE FUNCTION set_updated_at()
# RETURNS TRIGGER LANGUAGE plpgsql AS $$
# BEGIN
#   NEW.updated_at = NOW();
#   RETURN NEW;
# END;
# $$;

# CREATE TRIGGER trg_blood_reports_updated_at
# BEFORE UPDATE ON blood_reports
# FOR EACH ROW EXECUTE FUNCTION set_updated_at();

#### 4. PDF text extraction
Tries **Tabula** first (fast, for digital PDFs). Falls back to **Tesseract** for scanned PDFs.

In [31]:

def extract_text_tabula(pdf_path):
    """Extract text using pdfplumber — no Java needed."""
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # extract tables
            for table in page.extract_tables():
                for row in table:
                    text += " ".join(str(c) for c in row if c) + "\n"
            # extract plain text too
            text += page.extract_text() or ""
    return text, "pdfplumber"


def extract_text_tesseract(pdf_path):
    """OCR fallback for scanned PDFs."""
    images = convert_from_path(pdf_path, dpi=300)
    text   = " ".join(pytesseract.image_to_string(img) for img in images)
    return text, "tesseract"


def extract_text(pdf_path):
    """Try Tabula first; fall back to Tesseract if no content found."""
    text, method = extract_text_tabula(pdf_path)
    if len(text.strip()) < 50:          # too little text → probably scanned
        text, method = extract_text_tesseract(pdf_path)
    return text, method

#### 5. Parse blood markers from extracted text

In [ ]:
# # Each entry: (field_name, regex_pattern)
# MARKERS = [
#     ("glucose",           r"glucose[:\s]+([\d.]+)"),
#     ("hemoglobin",        r"h(?:a?e)?moglobin[:\s]+([\d.]+)"),
#     ("cholesterol_total", r"(?:total\s+)?cholesterol[:\s]+([\d.]+)"),
#     ("cholesterol_hdl",   r"hdl[:\s]+([\d.]+)"),
#     ("cholesterol_ldl",   r"ldl[:\s]+([\d.]+)"),
#     ("triglycerides",     r"triglycerides?[:\s]+([\d.]+)"),
#     ("wbc",               r"(?:wbc|white\s+blood\s+cells?)[:\s]+([\d.]+)"),
#     ("rbc",               r"(?:rbc|red\s+blood\s+cells?)[:\s]+([\d.]+)"),
# ]

# def parse_markers(text):
#     """Extract blood marker values from raw text using regex."""
#     text_lower = text.lower()
#     result = {}
#     for field, pattern in MARKERS:
#         match = re.search(pattern, text_lower)
#         result[field] = float(match.group(1)) if match else None
#     return result


# import re

# def parse_meta(text):
#     """Extract report date, lab name, and lab reference from text."""

#     date_match = re.search(
#         r"report\s*date[:\s]+([\d]{1,2}\s+[A-Za-z]+\s+\d{4})",
#         text, re.IGNORECASE
#     )

#     # fallback formats
#     if not date_match:
#         date_match = re.search(
#             r"report\s*date[:\s]+([\d]{1,2}[/-][\d]{1,2}[/-][\d]{2,4})",
#             text, re.IGNORECASE
#         )


#     lab_match = re.search(
#         r"(?:lab(?:oratory)?|clinic|hospital)[:\s]+([A-Za-z0-9 ]+)",
#         text, re.IGNORECASE
#     )
#     ref_match = re.search(
#     r"(?:lab\s*ref(?:erence)?|ref(?:erence)?\s*no|report\s*id)\s*[:\-]?\s*\n?\s*([A-Za-z0-9\-\/]+)",
#     text, re.IGNORECASE
#     )

 

#     report_date = date_match.group(1) if date_match else None
#     lab_name    = lab_match.group(1).strip() if lab_match else "Unknown"
#     lab_ref     = ref_match.group(1).strip() if ref_match else None

#     return report_date, lab_ref

In [32]:


# ── Markers aligned to every column in blood_reports ──────────────────────────

# CBC
UNIT_RE = r"(?:\s*\([^)]*\))?\s*"   # matches " (g/dL) " or nothing

MARKERS = [
    ("hemoglobin",        rf"h(?:a?e)?moglobin{UNIT_RE}([\d.]+)"),
    ("wbc",               rf"(?:wbc|white\s+blood\s+cells?){UNIT_RE}([\d.]+)"),
    ("rbc",               rf"(?:rbc|red\s+blood\s+cells?){UNIT_RE}([\d.]+)"),
    ("platelets",         rf"platelets?{UNIT_RE}([\d.]+)"),
    ("hematocrit",        rf"h(?:a?e)?matocrit{UNIT_RE}([\d.]+)"),
    ("mcv",               rf"mcv{UNIT_RE}([\d.]+)"),
    ("mch",               rf"mch\b{UNIT_RE}([\d.]+)"),
    ("mchc",              rf"mchc{UNIT_RE}([\d.]+)"),
    ("glucose",           rf"glucose{UNIT_RE}([\d.]+)"),
    ("hba1c",             rf"hb\s*a1c{UNIT_RE}([\d.]+)"),
    ("creatinine",        rf"creatinine{UNIT_RE}([\d.]+)"),
    ("bun",               rf"(?:bun|blood\s+urea\s+nitrogen){UNIT_RE}([\d.]+)"),
    ("egfr",              rf"e?gfr{UNIT_RE}([\d.]+)"),
    ("cholesterol_total", rf"(?:total\s+)?cholesterol{UNIT_RE}([\d.]+)"),
    ("cholesterol_hdl",   rf"hdl{UNIT_RE}([\d.]+)"),
    ("cholesterol_ldl",   rf"ldl{UNIT_RE}([\d.]+)"),
    ("triglycerides",     rf"triglycerides?{UNIT_RE}([\d.]+)"),
    ("alt",               rf"\balt\b{UNIT_RE}([\d.]+)"),
    ("ast",               rf"\bast\b{UNIT_RE}([\d.]+)"),
    ("bilirubin_total",   rf"(?:total\s+)?bilirubin{UNIT_RE}([\d.]+)"),
    ("tsh",               rf"\btsh\b{UNIT_RE}([\d.]+)"),

]


def parse_markers(text: str) -> dict:
    """Extract all blood marker values from raw text using regex."""
    text_lower = text.lower()
    result = {}
    for field, pattern in MARKERS:
        match = re.search(pattern, text_lower)
        result[field] = float(match.group(1)) if match else None
    return result


# ── Demographics ───────────────────────────────────────────────────────────────

def parse_demographics(text: str) -> dict:
    """Extract patient name, DOB, gender, age, and condition from report text."""

    name_match = re.search(
    r"patient\s*name[:\s]+([A-Za-z]+(?:\s+[A-Za-z]+){1,2})(?=\s+patient\s*id|\s+dob|\s+date|\s*$)",
    text, re.IGNORECASE
    )
    
    dob_match = re.search(
        r"(?:date\s+of\s+birth|dob)[:\s]+([\d]{1,2}\s+[A-Za-z]+\s+\d{4}|[\d]{1,2}[/-][\d]{1,2}[/-][\d]{2,4})",
        text, re.IGNORECASE
    )
    gender_match = re.search(
        r"gender[:\s]+(male|female|other)",
        text, re.IGNORECASE
    )
    age_match = re.search(
        r"age[:\s]+(\d+)",
        text, re.IGNORECASE
    )
    condition_match = re.search(
        r"condition[:\s]+([A-Za-z ]+)",
        text, re.IGNORECASE
    )

    # Normalise gender to match CHECK constraint
    raw_gender = gender_match.group(1).strip().lower() if gender_match else None
    gender = raw_gender if raw_gender in ("male", "female", "other") else None

    return {
        "patient_name":   name_match.group(1).strip()      if name_match      else None,
        "patient_dob":    parse_date(dob_match.group(1))   if dob_match       else None,
        "patient_gender": gender,
        "patient_age":    int(age_match.group(1))          if age_match       else None,
        "condition_note": condition_match.group(1).strip() if condition_match else None,
    }


# ── Report meta ────────────────────────────────────────────────────────────────

def parse_meta(text: str) -> dict:
    """Extract report_date and lab_ref from report text."""

    date_match = re.search(
        r"report\s*date[:\s]+([\d]{1,2}\s+[A-Za-z]+\s+\d{4})",
        text, re.IGNORECASE
    )
    if not date_match:
        date_match = re.search(
            r"report\s*date[:\s]+([\d]{1,2}[/-][\d]{1,2}[/-][\d]{2,4})",
            text, re.IGNORECASE
        )

    lab_match = re.search(
        r"(?:lab(?:oratory)?|clinic|hospital)[:\s]+([A-Za-z0-9 ]+)",
        text, re.IGNORECASE
    )
    ref_match = re.search(
        r"(?:lab\s*ref(?:erence)?|ref(?:erence)?\s*no|report\s*id)\s*[:\-]?\s*\n?\s*([A-Za-z0-9\-\/]+)",
        text, re.IGNORECASE
    )

    return {
        "report_date": parse_date(date_match.group(1)) if date_match else None,
        "lab_ref":     ref_match.group(1).strip()      if ref_match  else None,
    }


# ── Date helper ────────────────────────────────────────────────────────────────

def parse_date(raw: str):
    """Parse a date string into a Python date object."""
    formats = ["%d %B %Y", "%d %b %Y", "%d/%m/%Y", "%d-%m-%Y", "%d/%m/%y"]
    for fmt in formats:
        try:
            return datetime.strptime(raw.strip(), fmt).date()
        except ValueError:
            continue
    return None


# ── Master parser ──────────────────────────────────────────────────────────────

def parse_report(text: str, customer_id: str) -> dict:
    """
    Parse a full blood report text and return a dict ready
    to INSERT into the blood_reports table.
    """
    record = {"customer_id": customer_id}
    record.update(parse_meta(text))
    record.update(parse_demographics(text))
    record.update(parse_markers(text))
    return record

#### 6. Validate extracted values
Checks that values fall within medically plausible ranges.

In [ ]:
# # (min, max) plausible ranges — outside these = data quality issue
# RANGES = {
#     "glucose":           (20,   600),
#     "hemoglobin":        (3,    25),
#     "cholesterol_total": (50,   500),
#     "cholesterol_hdl":   (10,   150),
#     "cholesterol_ldl":   (10,   400),
#     "triglycerides":     (20,   2000),
#     "wbc":               (0.5,  100),
#     "rbc":               (1,    10),
# }

# def validate(markers):
#     """Return (is_valid, notes). Flags values outside plausible ranges."""
#     issues = []
#     for field, (lo, hi) in RANGES.items():
#         val = markers.get(field)
#         if val is not None and not (lo <= val <= hi):
#             issues.append(f"{field}={val} out of range [{lo}–{hi}]")
#     return (len(issues) == 0), "; ".join(issues) or None

In [33]:
# (min, max) plausible ranges — outside these = data quality issue
RANGES = {
    # CBC
    "hemoglobin":        (3.0,   25.0),
    "wbc":               (0.5,  100.0),
    "rbc":               (1.0,   10.0),
    "platelets":         (10.0, 1500.0),
    "hematocrit":        (5.0,   75.0),
    "mcv":               (50.0, 150.0),
    "mch":               (10.0,  50.0),
    "mchc":              (20.0,  45.0),

    # Metabolic / glucose
    "glucose":           (20.0,  600.0),
    "hba1c":             (2.0,   20.0),

    # Kidney
    "creatinine":        (0.1,   20.0),
    "bun":               (1.0,  150.0),
    "egfr":              (1.0,  200.0),

    # Lipid panel
    "cholesterol_total": (50.0,  500.0),
    "cholesterol_hdl":   (10.0,  150.0),
    "cholesterol_ldl":   (10.0,  400.0),
    "triglycerides":     (20.0, 2000.0),

    # Liver
    "alt":               (1.0,  2000.0),
    "ast":               (1.0,  2000.0),
    "bilirubin_total":   (0.1,   30.0),

    # Thyroid
    "tsh":               (0.01,  50.0),
}

# Clinical reference ranges per gender where they differ
CLINICAL_RANGES = {
    "male": {
        "hemoglobin":     (13.5, 17.5),
        "hematocrit":     (41.0, 53.0),
        "rbc":            (4.5,   5.9),
        "creatinine":     (0.6,   1.2),
        "cholesterol_hdl":(40.0, 999.0),
    },
    "female": {
        "hemoglobin":     (12.0, 15.5),
        "hematocrit":     (36.0, 46.0),
        "rbc":            (4.0,   5.2),
        "creatinine":     (0.5,   1.1),
        "cholesterol_hdl":(50.0, 999.0),
    },
}

# Severity thresholds — critical values that may need urgent flagging
CRITICAL_RANGES = {
    "glucose":       (50.0,  500.0),
    "hemoglobin":    (7.0,   20.0),
    "platelets":     (50.0, 1000.0),
    "wbc":           (2.0,   30.0),
    "creatinine":    (0.2,   10.0),
    "tsh":           (0.1,   20.0),
}


def validate(markers: dict, gender: str = None) -> tuple[bool, str | None, list[str]]:
    """
    Validate extracted marker values against plausible and clinical ranges.

    Args:
        markers : dict of {field: float|None} from parse_markers()
        gender  : 'male' | 'female' | None  — enables gender-adjusted ranges

    Returns:
        is_valid         : False if any plausible-range violation found
        validation_notes : semicolon-joined issue strings, or None
        critical_flags   : list of fields outside critical thresholds (for alerts)
    """
    issues   = []
    critical = []

    for field, (lo, hi) in RANGES.items():
        val = markers.get(field)
        if val is None:
            continue

        # 1. Plausible range check (data quality)
        if not (lo <= val <= hi):
            issues.append(f"{field}={val} outside plausible range [{lo}–{hi}]")
            continue                        # skip clinical check if already flagged

        # 2. Gender-adjusted clinical range check
        if gender and gender.lower() in CLINICAL_RANGES:
            clo, chi = CLINICAL_RANGES[gender.lower()].get(field, (None, None))
            if clo is not None and not (clo <= val <= chi):
                issues.append(
                    f"{field}={val} outside {gender} clinical range [{clo}–{chi}]"
                )

        # 3. Critical value check (regardless of clinical range)
        if field in CRITICAL_RANGES:
            crit_lo, crit_hi = CRITICAL_RANGES[field]
            if not (crit_lo <= val <= crit_hi):
                critical.append(f"{field}={val} CRITICAL [{crit_lo}–{crit_hi}]")

    is_valid = len(issues) == 0
    notes    = "; ".join(issues) if issues else None

    return is_valid, notes, critical


def validate_record(record: dict) -> dict:
    """
    Run validate() on a parsed report record and inject results back in-place.
    Convenience wrapper for use just before INSERT.

    Returns the same record dict with is_valid, validation_notes updated.
    """
    gender = record.get("patient_gender")
    is_valid, notes, critical = validate(record, gender=gender)

    record["is_valid"]         = is_valid
    record["validation_notes"] = notes

    if critical:
        existing = record.get("validation_notes") or ""
        record["validation_notes"] = (existing + " | CRITICAL: " + "; ".join(critical)).strip(" |")

    return record

#### 7. Save one report to PostgreSQL
`ON CONFLICT DO NOTHING` ensures **no duplicate records** are ever inserted.

In [ ]:
# INSERT_SQL = """
# INSERT INTO blood_reports (
#     customer_id, report_date, lab_name, lab_ref,
#     glucose, hemoglobin, cholesterol_total, cholesterol_hdl,
#     cholesterol_ldl, triglycerides, wbc, rbc,
#     raw_data, minio_path, extraction_method, is_valid, validation_notes
# ) VALUES (
#     %(customer_id)s, %(report_date)s, %(lab_name)s, %(lab_ref)s,
#     %(glucose)s, %(hemoglobin)s, %(cholesterol_total)s, %(cholesterol_hdl)s,
#     %(cholesterol_ldl)s, %(triglycerides)s, %(wbc)s, %(rbc)s,
#     %(raw_data)s, %(minio_path)s, %(extraction_method)s,
#     %(is_valid)s, %(validation_notes)s
# )
# ON CONFLICT (customer_id, report_date, lab_name, lab_ref) DO NOTHING
# """

# def save_report(pg, record):
#     with pg.cursor() as cur:
#         cur.execute(INSERT_SQL, record)
#         inserted = cur.rowcount == 1
#     pg.commit()
#     return inserted



In [34]:
import json
from psycopg2.extras import Json

INSERT_SQL = """
INSERT INTO blood_reports (
    -- identity
    customer_id, report_date, lab_ref,

    -- demographics
    patient_name, patient_dob, patient_gender, patient_age, condition_note,

    -- CBC
    hemoglobin, wbc, rbc, platelets, hematocrit, mcv, mch, mchc,

    -- metabolic
    glucose, hba1c,

    -- kidney
    creatinine, bun, egfr,

    -- lipid
    cholesterol_total, cholesterol_hdl, cholesterol_ldl, triglycerides,

    -- liver
    alt, ast, bilirubin_total,

    -- thyroid
    tsh,

    -- flexible + traceability
    raw_data, minio_path, extraction_method, is_valid, validation_notes
) VALUES (
    %(customer_id)s, %(report_date)s, %(lab_ref)s,

    %(patient_name)s, %(patient_dob)s, %(patient_gender)s, %(patient_age)s, %(condition_note)s,

    %(hemoglobin)s, %(wbc)s, %(rbc)s, %(platelets)s, %(hematocrit)s, %(mcv)s, %(mch)s, %(mchc)s,

    %(glucose)s, %(hba1c)s,

    %(creatinine)s, %(bun)s, %(egfr)s,

    %(cholesterol_total)s, %(cholesterol_hdl)s, %(cholesterol_ldl)s, %(triglycerides)s,

    %(alt)s, %(ast)s, %(bilirubin_total)s,

    %(tsh)s,

    %(raw_data)s, %(minio_path)s, %(extraction_method)s, %(is_valid)s, %(validation_notes)s
)
ON CONFLICT (customer_id, report_date, lab_ref) DO UPDATE SET
    patient_name      = EXCLUDED.patient_name,
    patient_dob       = EXCLUDED.patient_dob,
    patient_gender    = EXCLUDED.patient_gender,
    patient_age       = EXCLUDED.patient_age,
    condition_note    = EXCLUDED.condition_note,
    hemoglobin        = EXCLUDED.hemoglobin,
    wbc               = EXCLUDED.wbc,
    rbc               = EXCLUDED.rbc,
    platelets         = EXCLUDED.platelets,
    hematocrit        = EXCLUDED.hematocrit,
    mcv               = EXCLUDED.mcv,
    mch               = EXCLUDED.mch,
    mchc              = EXCLUDED.mchc,
    glucose           = EXCLUDED.glucose,
    hba1c             = EXCLUDED.hba1c,
    creatinine        = EXCLUDED.creatinine,
    bun               = EXCLUDED.bun,
    egfr              = EXCLUDED.egfr,
    cholesterol_total = EXCLUDED.cholesterol_total,
    cholesterol_hdl   = EXCLUDED.cholesterol_hdl,
    cholesterol_ldl   = EXCLUDED.cholesterol_ldl,
    triglycerides     = EXCLUDED.triglycerides,
    alt               = EXCLUDED.alt,
    ast               = EXCLUDED.ast,
    bilirubin_total   = EXCLUDED.bilirubin_total,
    tsh               = EXCLUDED.tsh,
    raw_data          = EXCLUDED.raw_data,
    minio_path        = EXCLUDED.minio_path,
    extraction_method = EXCLUDED.extraction_method,
    is_valid          = EXCLUDED.is_valid,
    validation_notes  = EXCLUDED.validation_notes,
    updated_at        = NOW()
WHERE
    -- only overwrite if new extraction has more data than existing row
    (
        EXCLUDED.hemoglobin IS NOT NULL OR
        EXCLUDED.glucose    IS NOT NULL OR
        EXCLUDED.hba1c      IS NOT NULL OR
        EXCLUDED.creatinine IS NOT NULL
    )
"""

# All known columns — any extra keys in record go into raw_data
KNOWN_FIELDS = {
    "customer_id", "report_date", "lab_ref",
    "patient_name", "patient_dob", "patient_gender", "patient_age", "condition_note",
    "hemoglobin", "wbc", "rbc", "platelets", "hematocrit", "mcv", "mch", "mchc",
    "glucose", "hba1c",
    "creatinine", "bun", "egfr",
    "cholesterol_total", "cholesterol_hdl", "cholesterol_ldl", "triglycerides",
    "alt", "ast", "bilirubin_total",
    "tsh",
    "raw_data", "minio_path", "extraction_method", "is_valid", "validation_notes",
}


def prepare_record(record: dict) -> dict:
    """
    Ensure every expected key exists (fills missing with None),
    and moves any unknown keys into raw_data as JSONB.
    """
    extra = {k: v for k, v in record.items() if k not in KNOWN_FIELDS}

    clean = {field: record.get(field) for field in KNOWN_FIELDS}

    # Merge extra keys into raw_data
    if extra:
        existing = clean.get("raw_data") or {}
        if isinstance(existing, str):
            existing = json.loads(existing)
        clean["raw_data"] = Json({**existing, **extra})
    elif clean.get("raw_data") is not None:
        clean["raw_data"] = Json(clean["raw_data"])

    return clean


def save_report(pg, record: dict) -> tuple[bool, str]:
    """
    Prepare and insert a parsed report record into blood_reports.

    Returns:
        inserted : True if new row, False if existing row was updated
        status   : 'inserted' | 'updated' | 'skipped'
    """
    clean = prepare_record(record)

    try:
        with pg.cursor() as cur:
            cur.execute(INSERT_SQL, clean)
            row_count = cur.rowcount
        pg.commit()

        if row_count == 1:
            return True, "inserted"
        else:
            return False, "skipped"

    except Exception as e:
        pg.rollback()
        raise RuntimeError(f"Failed to save report for {record.get('customer_id')}: {e}") from e

#### 8. Process one PDF end-to-end

In [ ]:
# BUCKET = "health-data"

# def process_pdf(minio, pg, object_path, customer_id):
#     tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
#     tmp_path = tmp.name
#     tmp.close()

#     try:
#         minio.fget_object(BUCKET, object_path, tmp_path)

#         text, method     = extract_text(tmp_path)
#         markers          = parse_markers(text)

#         report_date, lab, extracted_ref = parse_meta(text)
#         is_valid, notes  = validate(markers)

#         record = {
#             "customer_id":       customer_id,
#             "report_date":       report_date,
#             "lab_name":          lab,
#             "lab_ref":           object_path,   # ← unique file path as lab_ref
#             **markers,
#             "raw_data":          json.dumps(markers),
#             "minio_path":        object_path,
#             "extraction_method": method,
#             "is_valid":          is_valid,
#             "validation_notes":  notes,
#         }

#         inserted = save_report(pg, record)
#         status   = "✅ Saved" if inserted else "⏭️  Duplicate — skipped"
#         print(f"  {status}: {object_path}")

#     finally:
#         if os.path.exists(tmp_path):
#             os.unlink(tmp_path)



In [35]:
import os
import json
import tempfile

BUCKET = "health-data"

def process_pdf(minio, pg, object_path: str, customer_id: str) -> dict:
    """
    Full pipeline for a single PDF:
      1. Download from MinIO
      2. Extract text (tabula / tesseract)
      3. Parse meta, demographics, markers
      4. Validate (plausible + clinical + critical)
      5. Save to PostgreSQL

    Returns a status dict for logging / orchestration.
    """
    tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
    tmp_path = tmp.name
    tmp.close()

    try:
        # ── 1. Download ────────────────────────────────────────────────────────
        minio.fget_object(BUCKET, object_path, tmp_path)

        # ── 2. Extract text ────────────────────────────────────────────────────
        text, method = extract_text(tmp_path)

        # ── 3. Parse all sections ──────────────────────────────────────────────
        meta         = parse_meta(text)          # report_date,  lab_ref
        demographics = parse_demographics(text)  # patient_name, dob, gender, age, condition
        markers      = parse_markers(text)       # all 22 marker columns

        # ── 4. Validate ────────────────────────────────────────────────────────
        gender               = demographics.get("patient_gender")
        is_valid, notes, critical_flags = validate(markers, gender=gender)

        # Append critical flags to validation notes if any
        if critical_flags:
            critical_str = " | CRITICAL: " + "; ".join(critical_flags)
            notes = (notes or "") + critical_str

        # ── 5. Assemble record ─────────────────────────────────────────────────
        record = {
            # identity
            "customer_id":        customer_id,
            "lab_ref":            meta.get("lab_ref") or object_path,  # fallback to path
            **meta,
            **demographics,
            **markers,

            # traceability
            "raw_data":           markers,        # prepare_record() wraps as Json()
            "minio_path":         object_path,
            "extraction_method":  method,

            # validation
            "is_valid":           is_valid,
            "validation_notes":   notes,
        }

        # ── 6. Save ────────────────────────────────────────────────────────────
        inserted, status = save_report(pg, record)

        # ── 7. Log ─────────────────────────────────────────────────────────────
        icon = "✅" if status == "inserted" else ("♻️ " if status == "updated" else "⏭️ ")
        print(f"  {icon} {status.capitalize()}: {object_path}")

        if critical_flags:
            for flag in critical_flags:
                print(f"    ⚠️  {flag}")

        return {
            "object_path":    object_path,
            "customer_id":    customer_id,
            "status":         status,
            "is_valid":       is_valid,
            "critical_flags": critical_flags,
            "extraction":     method,
        }

    except Exception as e:
        pg.rollback()
        print(f"  ❌ Failed: {object_path} — {e}")
        return {
            "object_path":    object_path,
            "customer_id":    customer_id,
            "status":         "error",
            "error":          str(e),
            "is_valid":       False,
            "critical_flags": [],
            "extraction":     None,
        }

    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)

#### 9. Sync all blood reports from MinIO

Scans the entire bucket for PDFs under `blood-reports/` folders.
- ✅ New customer added to MinIO → picked up automatically
- ✅ New PDF added to existing customer → picked up automatically  
- ✅ Re-run anytime — duplicates are silently skipped

In [36]:
from collections import defaultdict
from datetime import datetime, timezone


def sync_all_blood_reports(minio, pg) -> dict:
    """
    Scan MinIO for all blood report PDFs and process each one.

    Expected path structure:
        customers/{customer_id}/blood-reports/{filename}.pdf

    Returns a summary dict with counts and any failed paths.
    """

    # ── 1. Discover PDFs ───────────────────────────────────────────────────────
    objects = minio.list_objects(BUCKET, prefix="customers/", recursive=True)
    pdfs = [
        obj for obj in objects
        if "blood-reports" in obj.object_name
        and obj.object_name.endswith(".pdf")
    ]

    total = len(pdfs)
    print(f"🔍 Found {total} blood report PDF(s) in MinIO\n")

    if total == 0:
        print("⚠️  No PDFs found — check bucket prefix or folder structure.")
        return {"total": 0, "inserted": 0, "updated": 0,
                "skipped": 0, "errors": 0, "failed_paths": []}

    # ── 2. Process each PDF ────────────────────────────────────────────────────
    counts       = defaultdict(int)   # inserted | updated | skipped | error
    failed_paths = []
    critical_reports = []

    start_time = datetime.now(timezone.utc)

    for i, obj in enumerate(pdfs, start=1):
        parts = obj.object_name.split("/")

        # Guard: skip malformed paths
        if len(parts) < 4:
            print(f"  ⚠️  Skipping malformed path: {obj.object_name}")
            counts["skipped"] += 1
            continue

        customer_id = parts[1]   # customers/{customer_id}/blood-reports/file.pdf

        print(f"[{i}/{total}] 👤 {customer_id}  →  {obj.object_name}")

        result = process_pdf(minio, pg, obj.object_name, customer_id)
        status = result.get("status", "error")
        counts[status] += 1

        if status == "error":
            failed_paths.append({
                "path":  obj.object_name,
                "error": result.get("error"),
            })

        if result.get("critical_flags"):
            critical_reports.append({
                "customer_id": customer_id,
                "path":        obj.object_name,
                "flags":       result["critical_flags"],
            })

    # ── 3. Summary ─────────────────────────────────────────────────────────────
    elapsed = (datetime.now(timezone.utc) - start_time).total_seconds()

    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅  Inserted  : {counts['inserted']}
♻️   Updated   : {counts['updated']}
⏭️   Skipped   : {counts['skipped']}
❌  Errors    : {counts['error']}
⏱️   Time      : {elapsed:.1f}s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")

    if failed_paths:
        print("\n⚠️  Failed paths:")
        for f in failed_paths:
            print(f"   • {f['path']} — {f['error']}")

    if critical_reports:
        print("\n🚨 Critical value alerts:")
        for r in critical_reports:
            print(f"   • {r['customer_id']}  {r['path']}")
            for flag in r["flags"]:
                print(f"       ↳ {flag}")

    print("\n🎉 Sync complete\n")

    return {
        "total":            total,
        "inserted":         counts["inserted"],
        "updated":          counts["updated"],
        "skipped":          counts["skipped"],
        "errors":           counts["error"],
        "elapsed_seconds":  elapsed,
        "failed_paths":     failed_paths,
        "critical_reports": critical_reports,
    }


# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    summary = sync_all_blood_reports(minio, pg)

🔍 Found 5 blood report PDF(s) in MinIO

[1/5] 👤 CUST_amara_patel_03630F04  →  customers/CUST_amara_patel_03630F04/blood-reports/BloodReport_CUST_ama_20260511.pdf
  ✅ Inserted: customers/CUST_amara_patel_03630F04/blood-reports/BloodReport_CUST_ama_20260511.pdf
[2/5] 👤 CUST_carlos_rivera_5A81755B  →  customers/CUST_carlos_rivera_5A81755B/blood-reports/BloodReport_CUST_car_20260511.pdf
  ✅ Inserted: customers/CUST_carlos_rivera_5A81755B/blood-reports/BloodReport_CUST_car_20260511.pdf
[3/5] 👤 CUST_fatima_alsayed_8594AA83  →  customers/CUST_fatima_alsayed_8594AA83/blood-reports/BloodReport_CUST_fat_20260511.pdf
  ✅ Inserted: customers/CUST_fatima_alsayed_8594AA83/blood-reports/BloodReport_CUST_fat_20260511.pdf
[4/5] 👤 CUST_john_whitfield_C4987FD5  →  customers/CUST_john_whitfield_C4987FD5/blood-reports/BloodReport_CUST_joh_20260511.pdf
  ✅ Inserted: customers/CUST_john_whitfield_C4987FD5/blood-reports/BloodReport_CUST_joh_20260511.pdf
[5/5] 👤 CUST_mei_lin_66CBFE0F  →  customers/CUST_mei_lin

In [27]:
# def sync_all_blood_reports(minio, pg):
#     """Scan MinIO for all blood report PDFs and process new ones."""
#     objects = minio.list_objects(BUCKET, prefix="customers/", recursive=True)
#     pdfs    = [
#         obj for obj in objects
#         if "blood-reports" in obj.object_name
#         and obj.object_name.endswith(".pdf")
#     ]

#     print(f"Found {len(pdfs)} blood report PDF(s) in MinIO\n")

#     for obj in pdfs:
#         parts       = obj.object_name.split("/")
#         customer_id = parts[1]            # customers/{customer_id}/blood-reports/file.pdf
#         print(f"👤 {customer_id}")
#         process_pdf(minio, pg, obj.object_name, customer_id)

#     print("\n🎉 Sync complete")


# # ▶️ Run the full sync
# sync_all_blood_reports(minio, pg)

#### 10.  Auto-sync on a schedule (optional)

In [ ]:
!pip install apscheduler --quiet

In [ ]:
from apscheduler.schedulers.background import BackgroundScheduler
from datetime import datetime, timezone

def scheduled_sync():
    print(f"\n⏰ Auto-sync at {datetime.now(timezone.utc).strftime('%H:%M UTC')}")
    sync_all_blood_reports(get_minio(), get_pg())

scheduler = BackgroundScheduler()
scheduler.add_job(scheduled_sync, "interval", minutes=30)  # ✏️ adjust interval
scheduler.start()
print("⏰ Scheduler started — syncing every 30 min. Run scheduler.shutdown() to stop.")

In [ ]:
# ⛔ Stop the scheduler
scheduler.shutdown()
print("Scheduler stopped.")

#### 11.  Verify — query the results

In [ ]:
def query(pg, sql, label):
    with pg.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    print(f"\n📊 {label}")
    print("  " + " | ".join(cols))
    print("  " + "-" * 70)
    for row in rows:
        print("  " + " | ".join(str(v) for v in row))

# All reports
query(pg,
    "SELECT customer_id, report_date, glucose, hemoglobin, is_valid FROM blood_reports ORDER BY customer_id",
    "All blood reports"
)

# Reports with validation issues
query(pg,
    "SELECT customer_id, report_date, validation_notes FROM blood_reports WHERE is_valid = FALSE",
    "Reports with validation issues"
)

# Count per customer
query(pg,
    "SELECT customer_id, COUNT(*) as reports FROM blood_reports GROUP BY customer_id",
    "Reports per customer"
)